# ⚡ Qwen-Image-2.1 Turbo Cloud Studio

Welcome! This notebook runs the state-of-the-art **Qwen-Image-2.1** model on Kaggle's free Tesla T4 GPU tier using **Viggle Turbo 8-step distillation** (INT8 ConvRot Visual DiT + Qwen3-VL 8B W4A8 Text Encoder). It enables high-speed photorealistic image generation (Text-to-Image) and image remixing (Image-to-Image) in under 10 seconds.

### **Instructions:**
1. In the Kaggle notebook settings (right sidebar), set **Accelerator** to **GPU T4** (1x or 2x T4).
2. Ensure **Internet** is turned **On**.
3. Run the cells in sequence.
4. Click the public **Cloudflare Tunnel URL** (`*.trycloudflare.com`) displayed at the bottom of the last cell to open the Gradio UI.
5. Keep this notebook tab open while generating images.

### 🛠️ Step 1: Environment Setup & Scratch Disk Redirect


In [ ]:
import os
import shutil

# Direct Hugging Face & Torch caches to /tmp (bypasses Kaggle's 20 GB disk limit)
os.environ["HF_HOME"] = "/tmp/huggingface"
os.environ["TORCH_HOME"] = "/tmp/torch"
os.environ["TMPDIR"] = "/tmp"

# Install system utilities and fast multi-threaded downloader
!apt-get update -qq && apt-get install -y -qq aria2 wget psmisc > /dev/null 2>&1
!pip install -q gradio huggingface_hub accelerate torchvision pillow

print("✓ Base tools and Gradio installed.")

### 📦 Step 2: Install ComfyUI & Setup Viggle Turbo Module


In [ ]:
import os
import shutil

# Clone ComfyUI into /tmp scratch disk
%cd /tmp
if not os.path.exists("/tmp/ComfyUI"):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
    %cd /tmp/ComfyUI
    !pip install -q -r requirements.txt

# Ensure output directory saves to Kaggle's working storage
os.makedirs("/kaggle/working/outputs", exist_ok=True)
if os.path.exists("/tmp/ComfyUI/output") and not os.path.islink("/tmp/ComfyUI/output"):
    shutil.rmtree("/tmp/ComfyUI/output")
if not os.path.exists("/tmp/ComfyUI/output"):
    os.symlink("/kaggle/working/outputs", "/tmp/ComfyUI/output")

# Install Viggle Turbo node and configure __init__.py module
os.makedirs("/tmp/ComfyUI/custom_nodes/viggle_turbo", exist_ok=True)
!wget -q -nc https://huggingface.co/Viggle/Qwen-Image-2.1-viggle-turbo/raw/main/comfyui/viggle_turbo.py \
    -O /tmp/ComfyUI/custom_nodes/viggle_turbo/viggle_turbo.py

init_code = """from .viggle_turbo import NODE_CLASS_MAPPINGS, NODE_DISPLAY_NAME_MAPPINGS
__all__ = ['NODE_CLASS_MAPPINGS', 'NODE_DISPLAY_NAME_MAPPINGS']
"""
with open("/tmp/ComfyUI/custom_nodes/viggle_turbo/__init__.py", "w") as f:
    f.write(init_code)

print("✓ ComfyUI and Viggle Turbo module configured.")

### 📥 Step 3: Fast Parallel Download of All Weights (~90 Seconds)


In [ ]:
import os
import subprocess

os.makedirs("/tmp/ComfyUI/models/diffusion_models", exist_ok=True)
os.makedirs("/tmp/ComfyUI/models/text_encoders", exist_ok=True)
os.makedirs("/tmp/ComfyUI/models/vae", exist_ok=True)

models = [
    # 1. INT8 ConvRot Visual DiT (7.26 GB)
    {
        "url": "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_int8_convrot.safetensors",
        "dir": "/tmp/ComfyUI/models/diffusion_models",
        "out": "qwen_image_2.1_int8_convrot.safetensors"
    },
    # 2. Qwen3-VL 8B Text Encoder W4A8 (6.31 GB)
    {
        "url": "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/text_encoders/qwen3vl_8b_w4a8.safetensors",
        "dir": "/tmp/ComfyUI/models/text_encoders",
        "out": "qwen3vl_8b_w4a8.safetensors"
    },
    # 3. Native RGBA VAE (676 MB)
    {
        "url": "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors",
        "dir": "/tmp/ComfyUI/models/vae",
        "out": "qwen_image_2.1_vae_bf16.safetensors"
    }
]

for m in models:
    dest_path = os.path.join(m["dir"], m["out"])
    if not os.path.exists(dest_path):
        print(f"Downloading {m['out']}...")
        cmd = f"aria2c -c -x 16 -s 16 -k 1M '{m['url']}' -d '{m['dir']}' -o '{m['out']}'"
        subprocess.run(cmd, shell=True, check=True)
    else:
        print(f"✓ {m['out']} ready.")

print("✓ All model weights verified.")

### ⚡ Step 4: Launch ComfyUI Daemon (Clean Port, Locks, & Healthcheck)


In [ ]:
import os
import subprocess
import time
import urllib.request

# 1. Kill any existing instances and clear port 8188
!fuser -k 8188/tcp > /dev/null 2>&1
!pkill -9 -f "main.py" > /dev/null 2>&1
!pkill -9 -f "python.*comfy" > /dev/null 2>&1
time.sleep(2)

# 2. Remove lingering SQLite database locks
db_base = "/tmp/ComfyUI/user/default/comfyui.db"
for ext in ["", "-wal", "-shm"]:
    lock_file = db_base + ext
    if os.path.exists(lock_file):
        try:
            os.remove(lock_file)
        except Exception:
            pass

# 3. Launch ComfyUI backend
log_file = open("/tmp/comfyui.log", "w")
comfy_proc = subprocess.Popen(
    [
        "python", "main.py",
        "--listen", "0.0.0.0",
        "--port", "8188",
        "--enable-cors-header", "*",
        "--disable-auto-launch"
    ],
    cwd="/tmp/ComfyUI",
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True
)

print("Starting ComfyUI backend on port 8188...")

# 4. Wait for server readiness
server_ready = False
for _ in range(45):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=1) as resp:
            if resp.status == 200:
                server_ready = True
                break
    except Exception:
        time.sleep(1)

if server_ready:
    print("🚀 ComfyUI backend is online and ready for generation!")
else:
    print("❌ Failed to start. Server logs:")
    with open("/tmp/comfyui.log", "r") as f:
        print("".join(f.readlines()[-30:]))

### 💻 Step 5: Launch Interactive Gradio UI


In [ ]:
import os
import subprocess
import time
import urllib.request
import urllib.error
import json
import random
import re
import gradio as gr
from PIL import Image

# ==========================================
# 1. AUTO-HEAL: Ensure ComfyUI Backend is Alive
# ==========================================
def is_comfy_alive():
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=1) as resp:
            return resp.status == 200
    except Exception:
        return False

if not is_comfy_alive():
    print("⚠️ ComfyUI is not running on port 8188. Auto-starting backend...")
    !fuser -k 8188/tcp > /dev/null 2>&1
    !pkill -9 -f "main.py" > /dev/null 2>&1
    time.sleep(1)
    
    db_base = "/tmp/ComfyUI/user/default/comfyui.db"
    for ext in ["", "-wal", "-shm"]:
        if os.path.exists(db_base + ext):
            try:
                os.remove(db_base + ext)
            except Exception:
                pass

    log_file = open("/tmp/comfyui.log", "w")
    subprocess.Popen(
        [
            "python", "main.py",
            "--listen", "0.0.0.0",
            "--port", "8188",
            "--enable-cors-header", "*",
            "--disable-auto-launch"
        ],
        cwd="/tmp/ComfyUI",
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True
    )
    
    for attempt in range(40):
        if is_comfy_alive():
            print("✓ ComfyUI backend is online!")
            break
        time.sleep(1)
    else:
        print("❌ ComfyUI failed to start. Tail of /tmp/comfyui.log:")
        with open("/tmp/comfyui.log", "r") as f:
            print("".join(f.readlines()[-30:]))
        raise RuntimeError("Could not connect to ComfyUI backend on port 8188.")
else:
    print("✓ ComfyUI backend is already running (warm cache active).")

# ==========================================
# 2. Setup Gradio, Input Dirs & Tunnel Ports
# ==========================================
!fuser -k 7860/tcp > /dev/null 2>&1
!pkill -f "cloudflared.*7860" > /dev/null 2>&1
time.sleep(1)

os.makedirs("/tmp/ComfyUI/input", exist_ok=True)
os.makedirs("/kaggle/working/outputs", exist_ok=True)

if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q -nc -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

# ==========================================
# 3. Resolution Helper Function
# ==========================================
def resolve_dimensions(preset, custom_w, custom_h):
    if preset == "Custom (Use sliders below)":
        w = int(round(custom_w / 64.0) * 64)
        h = int(round(custom_h / 64.0) * 64)
        return max(384, min(1536, w)), max(384, min(1536, h))
    
    res_map = {
        "⚡ Fast Draft 1:1 (512 x 512)": (512, 512),
        "⚡ Fast Draft 16:9 (640 x 384)": (640, 384),
        "1024 x 1024 (Square 1:1)": (1024, 1024),
        "1344 x 768 (Landscape 16:9 - YouTube / PC)": (1344, 768),
        "768 x 1344 (Portrait 9:16 - Reels / Shorts)": (768, 1344),
        "1216 x 832 (Landscape 3:2)": (1216, 832),
        "832 x 1216 (Portrait 2:3)": (832, 1216),
        "1152 x 896 (Landscape 4:3)": (1152, 896),
        "896 x 1152 (Portrait 3:4)": (896, 1152),
        "1536 x 640 (Cinematic 21:9)": (1536, 640)
    }
    return res_map.get(preset, (1024, 1024))

# ==========================================
# 4. Universal Pipeline Execution (Auto-Detects Any Output Node)
# ==========================================
def run_pipeline(workflow):
    req_data = json.dumps({"prompt": workflow}).encode("utf-8")
    req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=req_data, headers={"Content-Type": "application/json"})
    
    t_start = time.time()
    try:
        with urllib.request.urlopen(req) as resp:
            prompt_id = json.loads(resp.read().decode("utf-8"))["prompt_id"]
    except urllib.error.HTTPError as e:
        return None, f"❌ ComfyUI Error: {e.read().decode('utf-8')}"
    except Exception as e:
        return None, f"❌ Request Error: {str(e)}"
    
    # Poll for completion and find any SaveImage output dynamically
    while True:
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:8188/history/{prompt_id}") as resp:
                history = json.loads(resp.read().decode("utf-8"))
                if prompt_id in history:
                    outputs = history[prompt_id].get("outputs", {})
                    filename = None
                    for node_id, node_data in outputs.items():
                        if "images" in node_data and len(node_data["images"]) > 0:
                            filename = node_data["images"][0]["filename"]
                            break
                    
                    if filename:
                        elapsed = time.time() - t_start
                        out_path = f"/kaggle/working/outputs/{filename}"
                        return Image.open(out_path), f"✓ Completed in {elapsed:.1f}s"
        except Exception:
            pass
        time.sleep(1.0)

# ==========================================
# 5. Handler: Text to Image
# ==========================================
def generate_t2i(prompt, negative_prompt, steps, res_preset, custom_w, custom_h, seed, randomize_seed):
    if not is_comfy_alive():
        return None, "❌ Error: ComfyUI backend not responding. Re-run Cell 5."
    
    if randomize_seed or seed == -1:
        seed = random.randint(1, 9999999999)
    
    width, height = resolve_dimensions(res_preset, custom_w, custom_h)
    
    workflow = {
        "1": {"class_type": "UNETLoader", "inputs": {"unet_name": "qwen_image_2.1_int8_convrot.safetensors", "weight_dtype": "default"}},
        "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": "qwen3vl_8b_w4a8.safetensors", "type": "qwen_image"}},
        "3": {"class_type": "ModelSamplingAuraFlow", "inputs": {"model": ["1", 0], "shift": 3.0}},
        "4": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["2", 0], "text": prompt}},
        "5": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["2", 0], "text": negative_prompt if negative_prompt else "low quality, blurry, distorted"}},
        "6": {"class_type": "EmptySD3LatentImage", "inputs": {"width": width, "height": height, "batch_size": 1}},
        "7": {"class_type": "VAELoader", "inputs": {"vae_name": "qwen_image_2.1_vae_bf16.safetensors"}},
        "8": {
            "class_type": "KSampler",
            "inputs": {
                "model": ["3", 0], "positive": ["4", 0], "negative": ["5", 0],
                "latent_image": ["6", 0], "seed": seed, "steps": int(steps),
                "cfg": 1.0, "sampler_name": "res_multistep", "scheduler": "simple", "denoise": 1.0
            }
        },
        "9": {"class_type": "VAEDecode", "inputs": {"samples": ["8", 0], "vae": ["7", 0]}},
        "10": {"class_type": "SaveImage", "inputs": {"filename_prefix": "qwen21_t2i", "images": ["9", 0]}}
    }
    
    img, status = run_pipeline(workflow)
    if img:
        status += f" | {width}x{height} | Seed: {seed}"
    return img, status

# ==========================================
# 6. Handler: Image to Image (Remix / Variation)
# ==========================================
def generate_i2i(init_img, prompt, negative_prompt, denoise, steps, res_preset, custom_w, custom_h, seed, randomize_seed):
    if not is_comfy_alive():
        return None, "❌ Error: ComfyUI backend not responding. Re-run Cell 5."
    if init_img is None:
        return None, "❌ Please upload an image first."
        
    if randomize_seed or seed == -1:
        seed = random.randint(1, 9999999999)
        
    width, height = resolve_dimensions(res_preset, custom_w, custom_h)
    
    # Scale and save uploaded image into ComfyUI's input directory
    resized_img = init_img.convert("RGB").resize((width, height), Image.Resampling.LANCZOS)
    img_filename = f"i2i_input_{int(time.time())}.png"
    resized_img.save(f"/tmp/ComfyUI/input/{img_filename}")
    
    # Clean, cycle-free workflow graph
    workflow = {
        "1": {"class_type": "UNETLoader", "inputs": {"unet_name": "qwen_image_2.1_int8_convrot.safetensors", "weight_dtype": "default"}},
        "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": "qwen3vl_8b_w4a8.safetensors", "type": "qwen_image"}},
        "3": {"class_type": "ModelSamplingAuraFlow", "inputs": {"model": ["1", 0], "shift": 3.0}},
        "4": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["2", 0], "text": prompt}},
        "5": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["2", 0], "text": negative_prompt if negative_prompt else "low quality, blurry, distorted"}},
        "6": {"class_type": "LoadImage", "inputs": {"image": img_filename}},
        "7": {"class_type": "VAELoader", "inputs": {"vae_name": "qwen_image_2.1_vae_bf16.safetensors"}},
        "8": {"class_type": "VAEEncode", "inputs": {"pixels": ["6", 0], "vae": ["7", 0]}},
        "9": {
            "class_type": "KSampler",
            "inputs": {
                "model": ["3", 0], "positive": ["4", 0], "negative": ["5", 0],
                "latent_image": ["8", 0], "seed": seed, "steps": int(steps),
                "cfg": 1.0, "sampler_name": "res_multistep", "scheduler": "simple", "denoise": float(denoise)
            }
        },
        "10": {"class_type": "VAEDecode", "inputs": {"samples": ["9", 0], "vae": ["7", 0]}},
        "11": {"class_type": "SaveImage", "inputs": {"filename_prefix": "qwen21_i2i", "images": ["10", 0]}}
    }
    
    img, status = run_pipeline(workflow)
    if img:
        status += f" | {width}x{height} | Strength: {denoise} | Seed: {seed}"
    return img, status

# ==========================================
# 7. UI Layout
# ==========================================
ratio_choices = [
    "⚡ Fast Draft 1:1 (512 x 512)",
    "⚡ Fast Draft 16:9 (640 x 384)",
    "1024 x 1024 (Square 1:1)",
    "1344 x 768 (Landscape 16:9 - YouTube / PC)",
    "768 x 1344 (Portrait 9:16 - Reels / Shorts)",
    "1216 x 832 (Landscape 3:2)",
    "832 x 1216 (Portrait 2:3)",
    "1152 x 896 (Landscape 4:3)",
    "896 x 1152 (Portrait 3:4)",
    "1536 x 640 (Cinematic 21:9)",
    "Custom (Use sliders below)"
]

with gr.Blocks(title="Qwen-Image-2.1 Turbo Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ⚡ Qwen-Image-2.1 Turbo Studio")
    gr.Markdown("High-speed image generation & remixing running on Kaggle GPUs via **Viggle Turbo 8-step distillation**.")
    
    with gr.Tabs():
        # TAB 1: TEXT TO IMAGE
        with gr.Tab("🎨 Text to Image"):
            with gr.Row():
                with gr.Column(scale=4):
                    t2i_prompt = gr.Textbox(
                        label="Prompt", lines=3,
                        value="A futuristic neon city street in Tokyo during rain, cinematic reflections on wet asphalt, 8k photorealistic"
                    )
                    t2i_neg = gr.Textbox(label="Negative Prompt", value="")
                    t2i_res = gr.Dropdown(label="Aspect Ratio / Preset", choices=ratio_choices, value="1344 x 768 (Landscape 16:9 - YouTube / PC)")
                    
                    with gr.Accordion("⚙️ Custom Dimensions (Optional)", open=False):
                        t2i_cw = gr.Slider(512, 1536, value=1024, step=64, label="Custom Width")
                        t2i_ch = gr.Slider(512, 1536, value=1024, step=64, label="Custom Height")
                    
                    with gr.Row():
                        t2i_steps = gr.Slider(6, 12, value=8, step=1, label="Denoising Steps")
                        t2i_seed = gr.Number(label="Seed", value=42, precision=0)
                    t2i_rand = gr.Checkbox(label="Randomize Seed", value=True)
                    
                    t2i_btn = gr.Button("🚀 Generate Image", variant="primary", size="lg")
                    t2i_status = gr.Markdown("Ready to render.")
                    
                with gr.Column(scale=5):
                    t2i_out = gr.Image(label="Generated Output", type="pil", interactive=False)
                    
            t2i_btn.click(
                fn=generate_t2i,
                inputs=[t2i_prompt, t2i_neg, t2i_steps, t2i_res, t2i_cw, t2i_ch, t2i_seed, t2i_rand],
                outputs=[t2i_out, t2i_status]
            )

        # TAB 2: IMAGE TO IMAGE
        with gr.Tab("🖼️ Image to Image (Remix / Variations)"):
            with gr.Row():
                with gr.Column(scale=4):
                    i2i_input = gr.Image(label="Source Image", type="pil")
                    i2i_prompt = gr.Textbox(
                        label="Prompt", lines=3,
                        placeholder="Describe how to modify the image (e.g., Turn into an oil painting, add cyberpunk lighting...)"
                    )
                    i2i_neg = gr.Textbox(label="Negative Prompt", value="")
                    i2i_denoise = gr.Slider(
                        0.2, 0.95, value=0.65, step=0.05,
                        label="Denoising Strength (0.3 = Subtle Edit, 0.7 = Strong Transformation)"
                    )
                    i2i_res = gr.Dropdown(label="Output Aspect Ratio", choices=ratio_choices, value="1024 x 1024 (Square 1:1)")
                    
                    with gr.Accordion("⚙️ Custom Dimensions (Optional)", open=False):
                        i2i_cw = gr.Slider(512, 1536, value=1024, step=64, label="Custom Width")
                        i2i_ch = gr.Slider(512, 1536, value=1024, step=64, label="Custom Height")
                    
                    with gr.Row():
                        i2i_steps = gr.Slider(6, 12, value=8, step=1, label="Steps")
                        i2i_seed = gr.Number(label="Seed", value=42, precision=0)
                    i2i_rand = gr.Checkbox(label="Randomize Seed", value=True)
                    
                    i2i_btn = gr.Button("✨ Remix / Generate Variation", variant="primary", size="lg")
                    i2i_status = gr.Markdown("Upload an image to start.")
                    
                with gr.Column(scale=5):
                    i2i_out = gr.Image(label="Remixed Output", type="pil", interactive=False)
                    
            i2i_btn.click(
                fn=generate_i2i,
                inputs=[i2i_input, i2i_prompt, i2i_neg, i2i_denoise, i2i_steps, i2i_res, i2i_cw, i2i_ch, i2i_seed, i2i_rand],
                outputs=[i2i_out, i2i_status]
            )

demo.queue()
demo.launch(server_name="0.0.0.0", server_port=7860, share=False, inline=False, prevent_thread_lock=True)

# ==========================================
# 8. Cloudflare Tunnel
# ==========================================
cf_log = open("/tmp/cloudflared_gradio.log", "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"],
    stdout=cf_log,
    stderr=subprocess.STDOUT
)

print("Starting Cloudflare tunnel for Studio...")
tunnel_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists("/tmp/cloudflared_gradio.log"):
        with open("/tmp/cloudflared_gradio.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                tunnel_url = match.group(0)
                break

if tunnel_url:
    print("\n" + "="*60)
    print(f"🚀 QWEN 2.1 STUDIO READY: {tunnel_url}")
    print("="*60 + "\n")
else:
    print("Check /tmp/cloudflared_gradio.log for tunnel details.")